In [1]:
import sys
sys.path.append('../utils_function/')
import utils_function as uf
import common_functions as cf
import pandas as pd
import numpy as np
import os
from collections import defaultdict

In [2]:
def get_optimal_params(performance_dict):
    """
    从性能字典中找出平均相关性最高的参数
    
    参数:
    performance_dict (dict): 包含不同参数对应相关性列表的字典
    
    返回:
    float: 最优参数值
    """
    best_avg_corr = -float('inf')  # 初始化为负无穷

    for param, corr_list in performance_dict.items():
        avg_corr = sum(corr_list) / len(corr_list)
        std_corr = np.std(corr_list)
        # print(f"参数: {param}, 平均相关性: {avg_corr:.4f}, 标准差: {std_corr:.4f}")
        
        # 更新最优参数
        if avg_corr > best_avg_corr:
            best_avg_corr = avg_corr
            best_param = param

    return best_param

In [3]:
# animal_list = ['mouse', 'marmoset', 'macaque', 'human', 'marmoset']
# dataset_key_list = ['gamanut', 'majka', 'markov40', 'lariviere', 'V4']
animal_list = ['mouse', 'marmoset', 'macaque', 'human']
dataset_key_list = ['gamanut', 'majka', 'markov40', 'lariviere']
performance_dict = {}
optimal_params_dict = {}
for animal, dataset_key in zip(animal_list, dataset_key_list):
    print(f'Running for {animal} and {dataset_key}')

    performance_dict_this_animal = defaultdict(list)
    performance_dict_this_animal_with_seed = defaultdict(list)

    model_container = cf.MetaModelContainer(uf.AxonGrowthModel, cf.pj('../../results/para_search_betzel_like', f'{animal}_{dataset_key}'))

    for model in model_container:
        performance_dict_this_animal[(model.Lambda, model.gamma)].append(model.get_data_model_correlation())
        performance_dict_this_animal_with_seed[(model.Lambda, model.gamma, model.the_seed)].append(model.get_data_model_correlation())
    
    performance_dict[f'{animal}_{dataset_key}'] = performance_dict_this_animal
    # get_optimal_params(performance_dict_this_animal)
    best_Lambda, best_gamma, best_seed = get_optimal_params(performance_dict_this_animal_with_seed)
    optimal_params_dict[f'{animal}_{dataset_key}'] = {'Lambda': best_Lambda, 'gamma': best_gamma, 'the_seed': best_seed}
    print(performance_dict_this_animal_with_seed)
    print(f'Optimal lambda for {animal}_{dataset_key}: {optimal_params_dict[f"{animal}_{dataset_key}"]}')
    print(f'Optimal average correlation: {max([sum(v)/len(v) for v in performance_dict_this_animal_with_seed.values()])}')

Running for mouse and gamanut
defaultdict(<class 'list'>, {(0.9995647841678819, 0.4, 1): [0.4685969950670676], (0.5997582134266011, 0.4, 1): [0.5439358751158755], (0.7996614987972415, 0.4, 5): [0.4941468880817787], (0.2999032853706404, 0.4, 1): [0.5989722136523451], (0.5997582134266011, 0.4, 5): [0.5456337365918911], (0.4998065707412809, 0.4, 1): [0.5616862436391298], (0.3998549280559607, 0.4, 5): [0.5743105060282049], (0.5997582134266011, 0.4, 2): [0.5441077789552108], (0.9995647841678819, 0.4, 2): [0.48846824868914795], (0.7996614987972415, 0.4, 4): [0.499082919275227], (0.3998549280559607, 0.4, 1): [0.5779760792115471], (0.6997098561119213, 0.4, 4): [0.5227748048260638], (0.1, 0.4, 3): [0.5229163743434496], (0.2999032853706404, 0.4, 2): [0.597584352249106], (0.9995647841678819, 0.4, 5): [0.48663644444209014], (0.2999032853706404, 0.4, 5): [0.595994263266966], (0.8996131414825617, 0.4, 4): [0.49552312032031604], (0.8996131414825617, 0.4, 3): [0.49936369508080064], (0.1999516426853202

In [4]:
if cf.check_all_file_exist_with_any_extension('../../results/optimal_params_betzel_like/optimal_params_betzel_like'):
    print('already saved, please check whether you want to overwrite it')
else:
    cf.save_dict(optimal_params_dict, '../../results/optimal_params_betzel_like/optimal_params_betzel_like')